# Análisis de texto con PySpark: NLP y clustering de abstracts científicos

Los datos se pueden descargar  [aquí](https://www.kaggle.com/datasets/allen-institute-for-ai/CORD-19-research-challenge?resource=download)


## Análisis de texto con PySpark

En este notebook se desarrolla un pipeline básico de procesamiento de texto utilizando PySpark sobre un corpus de artículos científicos.

El objetivo es trabajar con datos textuales a gran escala, aplicando:
- técnicas de limpieza
- transformación 
- representación vectorial 
que permitan preparar el texto para tareas de análisis más avanzadas, como clustering o modelado semático.

Se utiliza un dataset de publicaciones científicas (CORD-19), lo que permite trabajar con texto real, no estructurado y con características propias de entornos de Big Data.


## Estructura del notebook
1. Instalación y carga de librerías  
2. Creación de la sesión Spark  
3. Carga del conjunto de datos  
4. Exploración inicial  
5. Limpieza y preparación del texto  
6. Tokenización y preprocesamiento NLP  
7. Vectorización TF-IDF  
8. Clustering con KMeans  
9. Interpretación de clusters  
10. Conclusiones

## 1. Instalación e importación de librerías

In [1]:
# !pip install pandas
# !pip install pyspark
# !pip install transformers
# !pip install sentence-transformers
# !pip install torch torchvision torchaudio
# !pip install kagglehub
# !pip install pyarrow
# !pip install ipywidgets

In [2]:
import os, kagglehub, json, torch
import pandas as pd
from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    size, col, length, lower, trim, regexp_replace,
    when, year, to_date, avg, explode
)

from pyspark.ml.feature import HashingTF, IDF, Tokenizer, StopWordsRemover
from pyspark.sql.types import StringType
from transformers import pipeline
from sentence_transformers import SentenceTransformer

from pyspark.ml.clustering import KMeans


## 2. Creación de la sesión Spark

En esta etapa se inicializa la sesión de Spark, que constituye el punto de entrada al entorno de procesamiento distribuido.

La sesión permite:

- crear y manipular DataFrames distribuidos
- ejecutar transformaciones sobre grandes volúmenes de datos
- aplicar algoritmos de machine learning del ecosistema Spark

In [3]:
spark = SparkSession.builder \
    .appName("NLP_PySpark") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.local.dir", "D:/spark-temp") \
    .getOrCreate()

## 3. Carga y preparación inicial de datos

El dataset es cargado en un DataFrame de PySpark, lo que permite trabajar de forma distribuida con grandes volúmenes de información.

A diferencia de pandas, PySpark permite escalar el procesamiento y aplicar transformaciones de forma eficiente sobre datasets que no cabrían en memoria local.

En esta etapa se seleccionan las columnas relevantes para el análisis textual.

In [ ]:
dataset_path = r"D:\repositorios\VSC\COVID_Pyspark\archive" #I have to change it to the path where I downloaded the dataset from Kaggle, which is "COVID-19 Open Research Dataset (CORD-19)". 
# You can download it from https://www.kaggle.com/datasets/allen-institute-for-ai/CORD-19-research-challenge. 
#After downloading and unzipping the dataset, you will find a folder named "archive" that contains the "metadata.csv" file, which we will use in this code.

metadata_path = os.path.join(dataset_path, "metadata.csv")

df_raw = spark.read.csv(
    metadata_path,
    header=True,
    inferSchema=True
)

## 4. Exploración inicial del dataset

Antes de aplicar transformaciones, se realiza una revisión general del conjunto de datos para comprender su estructura.

En esta etapa se inspeccionan:

- las columnas disponibles y los tipos de datos
- ejemplos de registros

Este nos permitirá verificar que los datos fueron cargados correctamente y orientar las decisiones de limpieza y preprocesamiento de datos.

In [5]:
print("\nEsquema:")
df_raw.printSchema()


Esquema:
root
 |-- cord_uid: string (nullable = true)
 |-- sha: string (nullable = true)
 |-- source_x: string (nullable = true)
 |-- title: string (nullable = true)
 |-- doi: string (nullable = true)
 |-- pmcid: string (nullable = true)
 |-- pubmed_id: string (nullable = true)
 |-- license: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- journal: string (nullable = true)
 |-- mag_id: string (nullable = true)
 |-- who_covidence_id: string (nullable = true)
 |-- arxiv_id: string (nullable = true)
 |-- pdf_json_files: string (nullable = true)
 |-- pmc_json_files: string (nullable = true)
 |-- url: string (nullable = true)
 |-- s2_id: string (nullable = true)



In [6]:
print("\nPrimeras filas:")
df_raw.show(5, truncate=80)


Primeras filas:
+--------+----------------------------------------+--------+--------------------------------------------------------------------------------+---------------------+--------+---------+-------+--------------------------------------------------------------------------------+------------+---------------------------------------------------------------+--------------+------+----------------+--------+----------------------------------------------------------------------+------------------------------------------+---------------------------------------------------+-----+
|cord_uid|                                     sha|source_x|                                                                           title|                  doi|   pmcid|pubmed_id|license|                                                                        abstract|publish_time|                                                        authors|       journal|mag_id|who_covidence_id|arxiv_id|                  

In [7]:
print("\nCantidad total de filas:")
print(df_raw.count())


Cantidad total de filas:
1056660


## 5. Limpieza y preparación de los datos

En esta sección se prepara el dataset para el análisis NLP.

Las tareas principales incluyen:

- selección de columnas relevantes,
- eliminación de registros incompletos,
- depuración de valores problemáticos.

El objetivo es obtener un DataFrame consistente, enfocado en el contenido textual y listo para su procesamiento posterior en PySpark.

Aqui vamos a seleccionar las columnas más relevantes para nuestro análisis 

In [8]:
df_selected = df_raw.select(
    "cord_uid",
    "title",
    "abstract",
    "journal",
    "authors",
    "url"
)

#### Filtrado de datos relevantes

Se eliminan registros que no contienen información textual clave, como título o abstract.

En el contexto de NLP, la calidad del texto es crítica: documentos incompletos o vacíos no aportan información semántica y pueden afectar negativamente las representaciones vectoriales y los modelos posteriores.


Aquí vamos a excluir las filas que tienen valores perdidos en las columnas `title` o `abstract`. A su vez, vamos a eliminar las filas que aparecen con valores duplicados en la variable `cord_uid`




In [9]:
df_clean = df_selected \
    .filter(col("title").isNotNull()) \
    .filter(col("abstract").isNotNull()) \
    .dropDuplicates(["cord_uid"])

print("Filas después de limpieza:", df_clean.count())

Filas después de limpieza: 764686


## 6. Limpieza y normalización del texto

Los campos textuales son procesados para asegurar consistencia y prepararlos para tareas posteriores de NLP.

Se aplican las siguientes transformaciones:

- **Conversión a minúsculas (`lower`)**: permite evitar duplicaciones artificiales de términos (por ejemplo, "COVID" y "covid" serían tratados como palabras distintas si no se normaliza).

- **Eliminación de espacios al inicio y al final (`trim`)**: limpia posibles inconsistencias generadas durante la carga de datos o el scraping.

- **Normalización de espacios internos (`regexp_replace`)**: se utiliza una expresión regular (`\s+`) para reemplazar múltiples espacios consecutivos por un único espacio.  
  Esto es importante porque en textos reales pueden aparecer saltos de línea, tabulaciones o espacios duplicados que afectan la tokenización.

- **Cálculo de longitud del texto (`length`)**: se generan variables adicionales (`abstract_length` y `title_length`) que permiten medir la cantidad de caracteres de cada texto.  
  Estas variables son útiles para filtrar documentos demasiado cortos y asegurar que el contenido tenga suficiente información para análisis posteriores.

Estos pasos reducen la variabilidad innecesaria del texto y mejoran la calidad de las representaciones que se construirán en etapas siguientes del pipeline.

In [11]:
df_clean = (
    df_clean
    .withColumn("title_clean", trim(lower(col("title"))))
    .withColumn("abstract_clean", trim(lower(col("abstract"))))
    .withColumn("abstract_clean", regexp_replace(col("abstract_clean"), r"\s+", " "))
    .withColumn("abstract_length", length(col("abstract")))
    .withColumn("title_length", length(col("title")))
)

In [12]:
df_clean.show(5, truncate=80)

+--------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+---------------+------------+
|cord_uid|                                                                           title|                                                                        abstract|                                                                         journal|                                                                         authors|                                                           

## Filtrado de abstracts para análisis NLP

Con el objetivo de trabajar con contenido textual suficientemente informativo, se filtran los documentos en función de la longitud del abstract.
**Importante:** Los abstracts demasiado cortos suelen contener poca información semántica y pueden introducir ruido en etapas posteriores del análisis. En este caso, se conservan únicamente aquellos documentos cuyo abstract tiene una longitud superior a 100 caracteres, utilizando la variable `abstract_length`.

En esta etapa se construye un subconjunto del dataset (`df_text_ready`) que contiene únicamente los documentos con suficiente contenido textual para ser utilizados en análisis de NLP.

In [13]:
df_text_ready = (
    df_clean
    .filter(col("abstract_length") > 100)
)

print(f"Rows ready for NLP: {df_text_ready.count()}")

df_text_ready.select("cord_uid", "title", "abstract_length").show(10, truncate=80)

Rows ready for NLP: 759230
+--------+--------------------------------------------------------------------------------+---------------+
|cord_uid|                                                                           title|abstract_length|
+--------+--------------------------------------------------------------------------------+---------------+
|000yfedk|Sleep Disturbances in Frontline Health Care Workers During the COVID-19 Pande...|           2561|
|0012g4t9|COVID-19 and mental health in Brazil: psychiatric symptoms in the general pop...|           1551|
|001lk86f|The Epidemic of Covid-19 in Africa: Demographic Effect, Under-Reporting of Ca...|           2236|
|002y5r4z|Nusinersen in pediatric and adult patients with type III spinal muscular atrophy|           1344|
|0036mel2|Effects of Gender Role Beliefs on Social Connectivity and Marital Safety: Fin...|           2194|
|0056tfa3|Differences in COVID-19 Vaccine Concerns Among Asian Americans and Pacific Is...|           1773|
|

In [14]:
df_text_ready.printSchema()

root
 |-- cord_uid: string (nullable = true)
 |-- title: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- journal: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- url: string (nullable = true)
 |-- title_clean: string (nullable = true)
 |-- abstract_clean: string (nullable = true)
 |-- abstract_length: integer (nullable = true)
 |-- title_length: integer (nullable = true)



Longitud promedio del abstract

In [15]:
df_text_ready.select(avg("abstract_length")).show()

+--------------------+
|avg(abstract_length)|
+--------------------+
|  1446.0432042990924|
+--------------------+



## Distribución de documentos por journal

Se analiza la cantidad de documentos por journal con el objetivo de identificar las principales fuentes de publicación dentro del dataset.

Este análisis permite comprender mejor la composición del corpus y detectar qué revistas concentran mayor volumen de publicaciones, lo cual puede influir en los patrones observados en etapas posteriores del análisis.


In [16]:
df_text_ready.groupBy("journal") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(20, truncate=80)

+--------------------------------------------+-----+
|                                     journal|count|
+--------------------------------------------+-----+
|                                        NULL|73073|
|                                     bioRxiv| 8942|
|                                    PLoS One| 8873|
|             Int J Environ Res Public Health| 8168|
|                                     Sci Rep| 5281|
|                                      Cureus| 3910|
|                               Front Psychol| 3283|
|                                    BMJ Open| 3276|
|                                     Viruses| 3098|
|                               Front Immunol| 3021|
|                              Sustainability| 2789|
|                         Front Public Health| 2772|
|                               Int J Mol Sci| 2292|
|                         Journal of virology| 2120|
|                                 J Med Virol| 1962|
|                                  J Clin Med|

## 7. Definición del conjunto de trabajo

Una vez preparado el texto, se define el subconjunto de columnas que será utilizado en las etapas posteriores del análisis.

Se conservan variables relevantes para:

- identificar cada documento
- analizar el texto
- describir estadísticamente los clusters obtenidos

Esto permite trabajar con un DataFrame más limpio y enfocado en el objetivo del proyecto.

In [17]:
df_analysis = df_text_ready.select(
    "cord_uid",
    "title",
    "abstract", # ver si borro
    "abstract_clean",
    "abstract_length", # ver si borro
    "journal"
)

df_analysis.show(5, truncate=80)

+--------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+---------------+--------------------------------------------------------------------------------+
|cord_uid|                                                                           title|                                                                        abstract|                                                                  abstract_clean|abstract_length|                                                                         journal|
+--------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+---------------+-----------------------------

## 8. Tokenización y eliminación de stopwords

En esta etapa se inicia el procesamiento estructurado del texto dentro del pipeline de NLP.

- **Tokenización (`Tokenizer`)**: el texto de cada abstract es dividido en tokens, es decir, en palabras individuales.  
  Este paso transforma el texto no estructurado en una secuencia de unidades que pueden ser procesadas por algoritmos.  
  La tokenización es fundamental, ya que define cómo se segmenta el lenguaje y afecta directamente las representaciones posteriores.

- **Eliminación de stopwords (`StopWordsRemover`)**: se eliminan palabras muy frecuentes como “the”, “and”, “of”, que aparecen en la mayoría de los documentos pero no aportan información relevante para distinguir temas.  
  Mantener estas palabras puede introducir ruido y reducir la capacidad del modelo para identificar términos realmente informativos.

El resultado es una representación más limpia del texto (`filtered_tokens`), centrada en términos con mayor contenido semántico, lo que mejora la calidad de las etapas posteriores como la vectorización (TF-IDF) y el clustering.

In [18]:
tokenizer = Tokenizer(
    inputCol="abstract_clean",
    outputCol="tokens"
)
df_tokens = tokenizer.transform(df_analysis)

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)
df_tokens = remover.transform(df_tokens)

df_tokens.select("title", "filtered_tokens").show(5, truncate=80)

+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                                           title|                                                                 filtered_tokens|
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|Sleep Disturbances in Frontline Health Care Workers During the COVID-19 Pande...|[background:, covid-19, pandemic,, health, care, workers, sharing, challenges...|
|COVID-19 and mental health in Brazil: psychiatric symptoms in the general pop...|[public, health, interventions, general, population, level, imperative, order...|
|The Epidemic of Covid-19 in Africa: Demographic Effect, Under-Reporting of Ca...|[background:, epidemic, covid-19, shown, different, development, africa, comp...|
|Nusinersen in p

## Filtrado de documentos sin contenido útil

Después de la eliminación de stopwords, algunos documentos pueden quedar sin tokens, ya que originalmente estaban compuestos únicamente por palabras muy frecuentes y poco informativas.

Estos documentos no aportan valor semántico al análisis y pueden generar problemas en etapas posteriores como la vectorización (TF-IDF) o el clustering.

Por este motivo, se filtran aquellos registros cuyo conjunto de tokens resultante está vacío, asegurando que todos los documentos contengan información relevante para el pipeline de NLP.

In [19]:
df_tokens_filtered = df_tokens.filter(size(col("filtered_tokens")) > 0)

In [20]:
df_tokens_filtered.count()

759230

## 9. Vectorización del texto con TF-IDF

Para poder aplicar algoritmos de machine learning, el texto debe transformarse en una representación numérica. Los modelos no pueden trabajar directamente con palabras, sino con vectores.

En esta sección se utiliza TF-IDF (Term Frequency – Inverse Document Frequency), una técnica que asigna un peso a cada término en función de su relevancia.

El proceso se realiza en dos etapas:

- **Cálculo de frecuencia de términos (`HashingTF`)**:  
  cada documento es transformado en un vector numérico donde cada posición representa un término.  
  En este caso se utiliza hashing, lo que permite trabajar de forma eficiente en entornos de Big Data sin necesidad de almacenar explícitamente todo el vocabulario.  
  El parámetro `numFeatures=1000` define la dimensionalidad del vector.

- **Reponderación por relevancia global (`IDF`)**:  
  se ajustan los pesos de los términos considerando su frecuencia en todo el corpus.  
  Las palabras muy comunes reciben menor peso, mientras que las más específicas adquieren mayor importancia.

El resultado final (`tfidf_features`) es una representación vectorial del texto que captura información semántica relevante y puede ser utilizada como entrada para algoritmos como clustering o clasificación.

<div style="background-color:#ffffff; color:#000000; padding:15px; border-radius:8px; border-left:5px solid #4CAF50;">

## ¿Qué significa representar texto como vectores?


En NLP, los textos no se pueden utilizar directamente en algoritmos de machine learning, ya que estos trabajan con datos numéricos. Por este motivo, es necesario transformar el texto en vectores. Este proceso se conoce como representación vectorial del texto.

La siguiente imagen ilustra este proceso utilizando una representación tipo *Bag of Words*:

<img src="https://maelfabien.github.io/assets/images/nlp_2.jpg" width="800"/>

En la imagen se puede observar cómo:

- En el **primer documento**, la palabra *"cat"* aparece **2 veces**, por lo que su valor en el vector es 2.
- En el **segundo documento**, aparecen palabras como *"food"*, *"eat"* y *"second"*, cada una con valor 1.
- En el **tercer documento**, aparecen términos como *"attack"*, *"bird"* y *"day"*, que no aparecen en los otros documentos.

Cada fila representa un documento y cada columna una palabra del vocabulario. Los valores indican la frecuencia de cada término.

Esta transformación es la base de la mayoría de los modelos de NLP tradicionales y permite:

- comparar documentos entre sí
- medir similitud entre textos
- aplicar modelos de machine learning

Aunque el ejemplo está en inglés, el mismo proceso aplica a cualquier idioma.

En este notebook, este enfoque se extiende mediante TF-IDF, donde además de la frecuencia, se incorpora la importancia relativa de cada término dentro del corpus.

</div>


In [21]:
hashingTF = HashingTF(
    inputCol="filtered_tokens",
    outputCol="raw_features",
    numFeatures=1000
)

df_tf = hashingTF.transform(df_tokens_filtered)

idf = IDF(
    inputCol="raw_features",
    outputCol="tfidf_features"
)
idf_model = idf.fit(df_tf)
df_tfidf = idf_model.transform(df_tf)

df_tfidf.select("title", "tfidf_features").show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Representación del texto con TF-IDF

El resultado mostrado corresponde a un vector TF-IDF en formato disperso.

Tiene la forma: `(1000, [índices], [valores])`

- 1000 indica el tamaño total del vector.
- Los índices representan las posiciones donde hay palabras relevantes.
- Los valores corresponden a la importancia (peso TF-IDF) de esas palabras.

Este formato no muestra todos los valores del vector, sino solo aquellos que son distintos de cero, lo que permite ahorrar memoria.

Cada documento queda representado como un conjunto de números que pueden ser utilizados en modelos de machine learning.

En este caso, no es posible ver directamente las palabras asociadas a cada posición, ya que se utilizó HashingTF, que transforma las palabras en índices numéricos.

## 10. Clustering de documentos con KMeans

Una vez obtenidas las representaciones vectoriales del texto, se aplica el algoritmo KMeans para agrupar documentos similares.

Este enfoque no supervisado permite identificar estructuras temáticas latentes en el corpus sin necesidad de etiquetas previas.

Dado que el análisis se ejecuta en Spark, se realiza una preparación adicional del DataFrame para asegurar una ejecución estable del algoritmo en entorno distribuido.

In [22]:
df_tfidf.select("title", "tfidf_features").show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Preparación de datos para clustering

Antes de aplicar KMeans, es necesario verificar que todos los documentos tengan información válida para ser representados como vectores. En este caso, se realiza el siguiente chequeo: `df_tokens_filtered.filter(size(col("filtered_tokens")) == 0).count()`

Este paso permite identificar cuántos documentos quedaron sin tokens luego del proceso de limpieza y eliminación de stopwords. Esto puede ocurrir cuando:
- el texto original es muy corto
- todas las palabras fueron eliminadas por ser stopwords
- el preprocesamiento eliminó demasiado contenido

Este chequeo es importante porque:
- los documentos sin tokens no pueden ser vectorizados correctamente
- pueden generar errores o resultados inconsistentes en TF-IDF o KMeans
- afectan la calidad del clustering

En caso de existir documentos vacíos, se recomienda filtrarlos antes de continuar con el pipeline. De esta forma, se asegura que el modelo trabaje únicamente con información relevante.

In [23]:
df_tokens_filtered.filter(size(col("filtered_tokens")) == 0).count()

0

## Aplicación de KMeans para clustering de documentos

Una vez que los textos han sido transformados en vectores TF-IDF, es posible aplicar algoritmos de machine learning. En este caso, se utiliza KMeans, un algoritmo de clustering no supervisado que agrupa documentos en función de su similitud.

Para facilitar la ejecución:
- se toma una muestra de 1000 documentos (`limit(1000)`) para reducir el costo computacional
- se ajusta el número de particiones (`repartition(2)`) para optimizar el procesamiento en Spark

El modelo se configura con:
- `featuresCol="tfidf_features"`: columna que contiene la representación numérica del texto
- `k=5`: número de clusters a generar
- `seed=42`: para asegurar reproducibilidad de los resultados

Finalmente, el modelo asigna a cada documento un cluster según su similitud con otros documentos.

In [24]:
df_analysis_sample = df_tfidf.limit(1000).repartition(2)

kmeans = KMeans(featuresCol="tfidf_features", predictionCol="cluster", k=5, seed=42)
model = kmeans.fit(df_analysis_sample)

df_clustered = model.transform(df_analysis_sample)
df_clustered.select("title", "cluster").show(10, truncate=80)

+--------------------------------------------------------------------------------+-------+
|                                                                           title|cluster|
+--------------------------------------------------------------------------------+-------+
|Evolving Virulence? Decreasing COVID-19 Complications among Massachusetts Hea...|      0|
|Disentangling the relationship between apolipoprotein E, cardiovascular disea...|      0|
|Systematic review and meta-analysis of the prevalence of common respiratory v...|      1|
|Sleep Disturbances in Frontline Health Care Workers During the COVID-19 Pande...|      1|
|Understanding, Verifying, and Implementing Emergency Use Authorization Molecu...|      0|
|Croatian pupils' perspectives on remote teaching and learning during the covi...|      0|
|Programmed translational frameshifting is likely required for expressions of ...|      0|
|How COVID-19 Transformed Problem-Based Learning at Carle Illinois College of ...|      0|

El resultado muestra cada documento junto con el cluster asignado por el modelo. Cada número de cluster representa un grupo de documentos con contenido similar.

Es importante tener en cuenta que:
- KMeans no conoce el significado del texto, solo agrupa en base a patrones numéricos
- los clusters no tienen una etiqueta explícita, deben interpretarse analizando los textos dentro de cada grupo
- documentos con temas similares tenderán a estar en el mismo cluster

Para interpretar correctamente los resultados, se recomienda:
- revisar varios títulos o abstracts dentro de cada cluster
- identificar temas comunes (por ejemplo: COVID-19, salud mental, educación, etc.)
- analizar si los grupos tienen coherencia temática

Este paso es clave para validar si el modelo está capturando correctamente la estructura del corpus.

## 11. Distribución de documentos por cluster

En esta etapa se analiza cómo se distribuyen los documentos entre los distintos clusters generados por KMeans. Este paso permite evaluar de forma inicial el comportamiento del modelo y detectar posibles problemas en la segmentación.

En particular, se busca:
- identificar si los clusters están balanceados
- detectar clusters muy pequeños o con pocos documentos
- verificar si el modelo está agrupando de manera razonable

Para ello, se calcula la cantidad de documentos asignados a cada cluster. Este análisis es importante porque una distribución muy desbalanceada puede indicar que:
- el número de clusters (`k`) no es adecuado
- existen temas dominantes en el corpus
- el modelo no está capturando correctamente la diversidad del contenido

In [25]:
df_clustered.groupBy("cluster").count().orderBy("cluster").show()

+-------+-----+
|cluster|count|
+-------+-----+
|      0|  860|
|      1|  130|
|      2|    2|
|      3|    7|
|      4|    1|
+-------+-----+



Los resultados muestran la cantidad de documentos asignados a cada cluster. Se observa una distribución fuertemente desbalanceada:
- un cluster concentra la mayoría de los documentos (por ejemplo, el cluster 0)
- otros clusters contienen muy pocos documentos

Este comportamiento puede indicar que:
- existe un tema dominante en el corpus (por ejemplo, COVID-19)
- los documentos son muy similares entre sí
- el valor de `k` podría no ser el más adecuado

Clusters muy pequeños (con pocos documentos) pueden representar:
- temas muy específicos o poco frecuentes
- ruido en los datos
- o agrupaciones poco relevantes

Este análisis inicial es clave para decidir los siguientes pasos, como:
- ajustar el número de clusters
- mejorar la representación del texto
- o analizar el contenido de cada cluster para validar su coherencia temática

## Análisis de términos por cluster

Para interpretar los clusters generados por KMeans, es necesario analizar qué palabras caracterizan a cada grupo. Dado que el modelo trabaja con representaciones numéricas (TF-IDF), no es posible interpretar directamente los clusters sin volver al texto.

En esta etapa se realiza el siguiente proceso:
- se toma una muestra de documentos para facilitar el análisis
- se separan los tokens de cada documento utilizando `explode`
- se agrupan las palabras por cluster
- se calcula la frecuencia de aparición de cada término dentro de cada cluster

Este procedimiento permite identificar los términos más representativos de cada grupo y facilita la interpretación temática de los clusters.

In [26]:
df_cluster_sample = df_clustered.limit(50)

df_cluster_words = (
    df_cluster_sample
    .select("cluster", explode(col("filtered_tokens")).alias("word"))
)

df_cluster_word_counts = (
    df_cluster_words
    .groupBy("cluster", "word")
    .count()
    .orderBy("cluster", col("count").desc())
)

df_cluster_word_counts.show(50, truncate=False)

+-------+-----------------+-----+
|cluster|word             |count|
+-------+-----------------+-----+
|0      |covid-19         |53   |
|0      |patients         |38   |
|0      |data             |26   |
|0      |health           |23   |
|0      |study            |23   |
|0      |pandemic         |21   |
|0      |sars-cov-2       |21   |
|0      |may              |18   |
|0      |viral            |17   |
|0      |care             |16   |
|0      |disease          |16   |
|0      |time             |16   |
|0      |infection        |16   |
|0      |using            |15   |
|0      |risk             |15   |
|0      |coronavirus      |15   |
|0      |treatment        |14   |
|0      |patient          |13   |
|0      |acute            |13   |
|0      |analysis         |12   |
|0      |respiratory      |12   |
|0      |impact           |12   |
|0      |positive         |11   |
|0      |medical          |11   |
|0      |current          |11   |
|0      |results          |11   |
|0      |thera

Los resultados muestran las palabras más frecuentes dentro de cada cluster.

Por ejemplo, en el cluster 0 se observan términos como:
- covid-19
- patients
- health
- pandemic
- sars-cov-2
- infection
- respiratory

Esto indica que el cluster está fuertemente asociado a contenido relacionado con COVID-19 y salud.

Este tipo de análisis permite:
- identificar el tema principal de cada cluster
- validar si los documentos fueron agrupados de forma coherente
- detectar posibles clusters dominantes en el corpus

También se observa la presencia de términos generales como "study", "data" o "analysis", que son frecuentes en artículos científicos y no necesariamente discriminan entre temas.

En conjunto, estos resultados sugieren que el modelo está capturando correctamente patrones temáticos relevantes, aunque podría beneficiarse de mejoras en el preprocesamiento o en la selección de features para lograr una mayor diferenciación entre clusters.

### Exploración cualitativa de los clusters

Hasta ahora analizamos los clusters a partir de las palabras más frecuentes.  
Sin embargo, para entender realmente qué representa cada grupo, es necesario observar ejemplos reales de documentos.

En esta sección, se muestran títulos y abstracts originales por cluster, lo que permite:

- validar si los clusters tienen coherencia semántica  
- identificar los temas principales de cada grupo  
- detectar posibles mezclas o ruido en los clusters  

Este análisis es clave para interpretar correctamente los resultados del clustering, más allá de métricas o frecuencias de palabras.

In [27]:
for c in range(5):
    print(f"\n=== Cluster {c} ===")
    df_clustered.filter(col("cluster") == c) \
        .select("title", "abstract_clean") \
        .show(5, truncate=120)


=== Cluster 0 ===
+-----------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------+
|                                                                                                            title|                                                                                                          abstract_clean|
+-----------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------+
|                  Dynamic recurrence risk and adjuvant chemotherapy benefit prediction by ctDNA in resected NSCLC|accurately evaluating minimal residual disease (mrd) could facilitate early intervention and personalized adjuvant th...|
|                Liver histopatho

A partir de los ejemplos de documentos, se pueden identificar patrones temáticos en cada cluster:

- **Cluster 0**  
  Predominan estudios clínicos y biomédicos relacionados con COVID-19, incluyendo diagnóstico, fisiopatología y técnicas experimentales.  
  También aparecen algunos temas más generales de salud, lo que sugiere un cluster amplio centrado en investigación médica.

- **Cluster 1**  
  Enfocado en aspectos clínicos y hospitalarios del COVID-19, como factores de riesgo, protocolos médicos y manejo de pacientes.  
  Se observa una fuerte orientación hacia estudios aplicados en contextos de salud.

- **Cluster 2**  
  Cluster pequeño y más específico, centrado en complicaciones respiratorias y sistemas de salud.  
  Puede representar un subtema muy concreto dentro del corpus.

- **Cluster 3**  
  Incluye estudios sobre impacto social, comportamiento, alimentación y contextos familiares durante la pandemia.  
  Representa una dimensión más social y conductual del COVID-19.

- **Cluster 4**  
  Muy reducido y altamente específico, enfocado en análisis genómico y métodos computacionales aplicados al virus.  
  Posiblemente un cluster muy especializado o con pocos documentos.

**conclusiones generales**

- Los clusters capturan distintas dimensiones del fenómeno COVID-19:
  - clínica/médica  
  - hospitalaria  
  - social  
  - técnica/genómica  

- Existe cierto desbalance en el tamaño de los clusters, lo que sugiere que algunos temas son mucho más frecuentes en el dataset.

- En general, el modelo logra separar razonablemente bien los principales temas, aunque algunos clusters pueden ser demasiado amplios o contener ruido.

## Estadísticos descriptivos por cluster

Además del análisis de palabras y ejemplos de documentos, es útil explorar características cuantitativas de cada cluster.

En esta sección se calcula la longitud promedio de los abstracts por cluster, lo que permite:

- identificar diferencias estructurales entre grupos de documentos  
- detectar clusters con textos más largos o más concisos  
- complementar la interpretación semántica con información descriptiva  

Este tipo de análisis ayuda a entender no solo de qué tratan los clusters, sino también cómo están construidos los textos dentro de cada grupo.

In [28]:
df_clustered.groupBy("cluster") \
    .agg(avg("abstract_length").alias("avg_length")) \
    .orderBy("cluster") \
    .show()

+-------+------------------+
|cluster|        avg_length|
+-------+------------------+
|      0|1274.8046511627906|
|      1|            2365.1|
|      2|            2278.0|
|      3|2031.4285714285713|
|      4|            2246.0|
+-------+------------------+



Los resultados muestran diferencias claras en la longitud promedio de los abstracts entre clusters:

- **Cluster 0 (~1275 palabras)**  
  Presenta abstracts significativamente más cortos en comparación con el resto.  
  Esto puede indicar documentos más concisos, resúmenes breves o textos menos detallados.

- **Clusters 1, 2 y 4 (~2200–2365 palabras)**  
  Tienen abstracts más largos y relativamente similares entre sí.  
  Esto sugiere documentos más completos, posiblemente estudios clínicos o artículos científicos detallados.

- **Cluster 3 (~2031 palabras)**  
  Se encuentra en un punto intermedio, con textos relativamente extensos pero algo más cortos que los clusters principales.

### Conclusiones

- Existe una clara separación estructural entre clusters, no solo temática sino también en longitud del texto.  
- El Cluster 0 podría representar un tipo distinto de documentos (por ejemplo, resúmenes más breves o formatos diferentes).  
- Los clusters más grandes tienden a contener textos más extensos, lo que podría estar relacionado con mayor complejidad o profundidad del contenido.

Este análisis complementa la interpretación previa, mostrando que los clusters capturan tanto diferencias temáticas como estructurales.

## Distribución de documentos por cluster

En esta sección se analiza la cantidad de documentos asignados a cada cluster.

Este análisis es fundamental para:

- entender cómo se distribuyen los datos entre los clusters  
- detectar posibles desbalances (clusters muy grandes o muy pequeños)  
- evaluar la calidad del clustering  

Un buen clustering suele evitar grupos extremadamente dominantes o clusters con muy pocos elementos, ya que esto puede indicar problemas en la segmentación.

In [29]:
df_clustered.groupBy("cluster").count().show()

+-------+-----+
|cluster|count|
+-------+-----+
|      1|  130|
|      3|    7|
|      0|  860|
|      4|    1|
|      2|    2|
+-------+-----+



Los resultados muestran un fuerte desbalance en la cantidad de documentos por cluster:

- **Cluster 0 (860 documentos)**  
  Es claramente dominante, concentrando la gran mayoría de los datos.  
  Esto sugiere que el modelo está agrupando muchos documentos bajo un mismo tema amplio.

- **Cluster 1 (130 documentos)**  
  Representa un grupo secundario relevante, aunque mucho más pequeño que el cluster principal.

- **Clusters 2, 3 y 4 (2, 7 y 1 documentos respectivamente)**  
  Son extremadamente pequeños, lo que puede indicar:
  - subtemas muy específicos  
  - ruido en los datos  
  - o una segmentación poco equilibrada  

### Conclusiones

- El clustering presenta un **alto nivel de desbalance**, con un cluster dominante y varios clusters muy pequeños.  
- Esto puede indicar que:
  - el número de clusters (k=5) no es el más adecuado  
  - o que los datos tienen una estructura muy concentrada en pocos temas  

- Sería recomendable explorar:
  - otros valores de k  
  - o métodos adicionales para validar la calidad del clustering  

Este análisis es clave para interpretar correctamente los resultados y evaluar posibles mejoras en el modelo.

## Distribución de documentos por journal dentro de cada cluster

En esta sección se analiza cómo se distribuyen los documentos según el journal de publicación dentro de cada cluster.

Este análisis permite:

- identificar si ciertos clusters están dominados por journals específicos  
- detectar patrones de publicación asociados a determinados temas  
- evaluar si los clusters reflejan comunidades científicas o áreas de investigación concretas  

Además, esta información ayuda a complementar la interpretación temática, conectando los clusters con sus fuentes de origen.

In [30]:
df_clustered.groupBy("cluster", "journal") \
    .count() \
    .orderBy("cluster", col("count").desc()) \
    .show(20, truncate=False)

+-------+-------------------------------+-----+
|cluster|journal                        |count|
+-------+-------------------------------+-----+
|0      |NULL                           |77   |
|0      |bioRxiv                        |12   |
|0      |PLoS One                       |9    |
|0      |Sci Rep                        |8    |
|0      |Int J Environ Res Public Health|6    |
|0      |Vaccines (Basel)               |5    |
|0      |Front Immunol                  |5    |
|0      |Front Psychol                  |5    |
|0      |Journal of virology            |4    |
|0      |Emerg Infect Dis               |4    |
|0      |BMJ Open                       |4    |
|0      |Viruses                        |4    |
|0      |Int J Infect Dis               |3    |
|0      |Ann Clin Transl Neurol         |3    |
|0      |J Clin Microbiol               |3    |
|0      |Heliyon                        |3    |
|0      |Foods                          |3    |
|0      |Access Microbiol               

Los resultados muestran cómo se agrupan los documentos por journal dentro de cada cluster, con algunos patrones relevantes:

- **Alta presencia de valores NULL**  
  Una proporción importante de documentos no tiene información de journal.  
  Esto puede deberse a datos incompletos o a fuentes como preprints sin asignación formal.

- **Presencia de repositorios de preprints (bioRxiv, medRxiv)**  
  Estos aparecen entre los más frecuentes, lo que es consistente con la rápida difusión científica durante la pandemia.  
  Indica que el dataset incluye investigación en etapas tempranas.

- **Diversidad de journals en un mismo cluster**  
  En lugar de estar dominados por un único journal, los clusters contienen publicaciones de múltiples fuentes.  
  Esto sugiere que el clustering está capturando temas transversales, no limitados a una sola revista.

- **Presencia de journals especializados**  
  Revistas como *Journal of Virology*, *Emerging Infectious Diseases* o *Vaccines* indican un fuerte componente biomédico y epidemiológico.

**Conclusiones**

- Los clusters no están definidos por el journal, sino por el contenido temático de los documentos.  
- Existe una mezcla de publicaciones revisadas por pares y preprints, lo que refleja la naturaleza dinámica del corpus.  
- La presencia de valores NULL sugiere que podría ser útil realizar un análisis adicional o limpieza de esta variable.

Este análisis aporta contexto adicional y refuerza la interpretación de los clusters desde una perspectiva de origen de datos.

## Liberación de recursos computacionales

Una vez finalizado el análisis, es importante liberar los recursos utilizados durante la sesión de Spark.

Spark mantiene procesos activos en memoria y en el sistema, por lo que cerrar correctamente la sesión ayuda a:

- liberar memoria (RAM)  
- finalizar procesos en segundo plano  
- evitar consumo innecesario de recursos  

Para ello, se ejecuta el siguiente comando, que detiene la sesión de Spark:

```python
spark.stop()

In [31]:
spark.stop()

## **Conclusiones finales**

En este trabajo se aplicaron técnicas de procesamiento de lenguaje natural (NLP) y clustering no supervisado sobre un conjunto de abstracts científicos relacionados con COVID-19.

### Principales hallazgos

- **Identificación de temas principales**  
  El modelo de clustering (K-Means) permitió agrupar los documentos en distintos clusters que reflejan áreas temáticas diferenciadas:
  - investigación clínica y biomédica  
  - gestión hospitalaria y factores de riesgo  
  - impacto social y conductual  
  - análisis genómico y técnico  

- **Interpretabilidad de los clusters**  
  La combinación de:
  - palabras más frecuentes  
  - ejemplos reales de documentos  
  permitió validar que los clusters tienen coherencia semántica y representan temas reconocibles.

- **Desbalance en la distribución**  
  Se observó un cluster dominante que concentra la mayoría de los documentos, junto con clusters muy pequeños.  
  Esto sugiere que:
  - los datos están fuertemente concentrados en ciertos temas  
  - o que el número de clusters (k) podría ajustarse para mejorar la segmentación  

- **Diferencias estructurales entre clusters**  
  El análisis de longitud de abstracts mostró variaciones relevantes, indicando que los clusters no solo difieren en contenido, sino también en estructura textual.

- **Independencia del journal**  
  La distribución por journal evidenció que los clusters no están determinados por la fuente de publicación, sino por el contenido de los textos.  
  Además, se observó la presencia de preprints (bioRxiv, medRxiv), lo que refleja la dinámica de publicación durante la pandemia.

### Limitaciones

- El uso de **K-Means** implica asumir clusters de forma esférica, lo que puede no capturar completamente la complejidad semántica del texto.  
- La elección de **k=5** no fue optimizada mediante métricas como silhouette score o elbow method.  
- La representación basada en **TF-IDF** no captura contexto semántico profundo (a diferencia de embeddings).

### Posibles mejoras

- Evaluar diferentes valores de k y métricas de calidad de clustering  
- Incorporar representaciones más avanzadas (por ejemplo, embeddings con transformers)  
- Aplicar técnicas de reducción de dimensionalidad para visualización (PCA, t-SNE)  
- Mejorar el preprocesamiento (detección de entidades, normalización avanzada)

### Conclusión general

El enfoque aplicado permite segmentar de forma efectiva un corpus de textos científicos, identificando patrones temáticos relevantes y proporcionando una base sólida para análisis exploratorios en contextos de NLP aplicado.

Este tipo de análisis es especialmente útil en escenarios como:
- exploración de literatura científica  
- organización automática de documentos  
- generación de insights en grandes volúmenes de texto  